# Part 11 - Shipping It: tracing, cost-per-success, and the core capstone

*Agents from First Principles, Part 11.*

This is the last part of the core track. Over ten parts we ringed a bare LLM into an agent that can plan, recover, remember, stop itself, survive a crash, and ask a human. There is one thing left before it is shippable: you cannot see into it. A durable agent you cannot observe is undebuggable and unaccountable. When it is slow, you do not know which step. When it is expensive, you do not know which call. When it "works," you cannot prove it worked for the right reasons or at an acceptable cost.

The good news is that Part 9 already did the hard part. The event journal is a complete record of the run. **Observability is not a second system; it is a SECOND VIEW of the same log.** This part folds the journal into spans, reconstructs the metrics that matter from those spans, and closes the core track with a capstone.

1. **A span tree, OTel-GenAI-shaped, by hand (no otel SDK).** The run becomes one root span `invoke_agent`, with child spans for each llm decision (operation `chat`) and each `execute_tool` call. Each span carries `gen_ai.*`-STYLE attributes: `operation.name`, `request.model`, `tool.name`, usage input/output tokens, and latency. We emit them as JSONL.

2. **Metrics reconstructed from the spans:** the trajectory (the tool sequence), total tokens and cost, per-step latency, and the one number that matters in production, COST PER SUCCESS.

3. **One log, two views.** We print the same journal as the durability artifact (Part 9) AND as the span tree, to make the point that they are the same bytes read two ways.

4. **The capstone.** A Part 1 to Part 11 checklist: every ring we added to the augmented LLM, from a bare loop to a production-shaped core agent.

> **Runs with no network, no API key, and no third-party dependency.** The frozen journal is the offline source of truth; `generate()` (the real LLM call) and a real OTLP exporter are each one flag away. Set `OPENAI_API_KEY` to see the real banner; the code always folds the frozen journal so output stays reproducible.

Companion script: `observable_agent.py`

## Setup

Two standard-library imports do the work: `json` serializes the journal and the spans (the same bytes, two views), and `os` lets us check for an API key without ever requiring one. Everything else is plain Python, so every cell runs fully offline, exactly as in Parts 1-10.

`PRICE_PER_TOKEN` is the illustrative price carried from Part 8's budget accounting. It is the single knob that turns token usage into dollars, both in the root span's `gen_ai.cost_usd` and in the cost-per-success table.

In [ ]:
import json
import os

PRICE_PER_TOKEN = 0.00001                  # illustrative, from Part 8

print("ready -- no network, no API key, no third-party dependency required")

## Step 1 - The journal for one refund run, annotated for spans

This is the Part 9/10 journal, REUSED, with two extra fields per event that a span needs: a latency (`ms`) and, on each llm decision, the token counts (`in` / `out`). The numbers are frozen so the spans are byte-reproducible. The run is the familiar refund for `ORD-3300`, `run_id` `run-bb11`: search the policy, process the refund, then finish.

**Sidebar - transcript economics.** Watch the `in` field grow across the three llm decisions: `40 -> 70 -> 95`. The agent re-sends a growing transcript every step (the goal, plus every prior Thought, Action, and Observation), so input tokens climb monotonically. That growth, not the tool calls, is the dominant cost lever as a run gets longer. The lever back is prompt/KV caching: cache the unchanged prefix so you are not billed full price for re-sending it. The `out` tokens (what the model writes) stay small; it is the input you keep paying to re-read.

In [ ]:
# llm decisions carry tokens (in/out) and latency (ms); tool results carry latency.
# Note the input_tokens GROWING (40 -> 70 -> 95): the transcript re-sent every step.
JOURNAL = [
    {"seq": 0, "type": "run_started", "data": {"run_id": "run-bb11", "goal": "refund ORD-3300"}},
    {"seq": 1, "type": "llm_decided", "data": {"tool": "search_policy", "in": 40, "out": 12, "ms": 180}},
    {"seq": 2, "type": "tool_result", "data": {"tool": "search_policy", "ms": 60,
                                               "result": "refunds after the window: 10% restocking fee"}},
    {"seq": 3, "type": "llm_decided", "data": {"tool": "process_refund", "in": 70, "out": 15, "ms": 210}},
    {"seq": 4, "type": "tool_result", "data": {"tool": "process_refund", "ms": 90,
                                               "result": "refunded $180.00 to ORD-3300"}},
    {"seq": 5, "type": "llm_decided", "data": {"tool": "finish", "in": 95, "out": 20, "ms": 150}},
    {"seq": 6, "type": "finished", "data": {"answer": "Refund of $180.00 for ORD-3300 is complete."}},
]

print(f"journal: {len(JOURNAL)} events; run_id run-bb11; input tokens 40 -> 70 -> 95")

## Step 2 - Fold the journal into an OTel-GenAI-shaped span tree (no otel SDK)

This is the heart of the part. `build_spans` walks the journal once and produces a list of plain dicts: one root span `invoke_agent`, then a child span per step, as siblings under the root. Each `llm_decided` becomes an `llm` span with operation `chat`; each `tool_result` becomes an `execute_tool` span. A running `clock` lays the children end to end, so the root's `end_ms` is the total latency, and the root aggregates tokens and cost.

**No otel SDK, on purpose.** These are just dicts we can emit as JSONL. We avoid the SDK dependency for two reasons: it would break the offline, zero-dependency bar this whole series holds; and the attribute names are still moving. The GenAI semantic conventions are not frozen, so we hedge every attribute as `gen_ai.*`-STYLE rather than claim conformance. A real OTLP exporter is the same swap-in pattern as `generate()`: one line. Where this code builds a dict, a real exporter would open a span instead:

```python
# from opentelemetry import trace
# tracer = trace.get_tracer("agent")
# with tracer.start_as_current_span("invoke_agent") as root:
#     root.set_attribute("gen_ai.operation.name", "invoke_agent")
#     ...
```

In [ ]:
def build_spans(journal):
    """Fold the journal into spans: invoke_agent root + one child per step.
    llm_decided -> llm (chat) span; tool_result -> execute_tool span. No otel SDK:
    plain dicts we can emit as JSONL. A real exporter is a commented one-liner."""
    spans, clock, sid = [], 0, 0

    def new_span(name, attrs, start, end):
        nonlocal sid
        sid += 1
        return {"span_id": f"s{sid}", "parent_id": "s0", "name": name,
                "start_ms": start, "end_ms": end, "attributes": attrs}

    children = []
    for e in journal:
        d = e["data"]
        if e["type"] == "llm_decided":
            attrs = {"gen_ai.operation.name": "chat",
                     "gen_ai.request.model": "deterministic-rule-controller",
                     "gen_ai.usage.input_tokens": d["in"],
                     "gen_ai.usage.output_tokens": d["out"],
                     "decided.tool": d["tool"]}
            children.append(new_span("llm", attrs, clock, clock + d["ms"]))
            clock += d["ms"]
        elif e["type"] == "tool_result":
            attrs = {"gen_ai.operation.name": "execute_tool", "gen_ai.tool.name": d["tool"]}
            children.append(new_span("execute_tool", attrs, clock, clock + d["ms"]))
            clock += d["ms"]

    in_tok = sum(e["data"]["in"] for e in journal if e["type"] == "llm_decided")
    out_tok = sum(e["data"]["out"] for e in journal if e["type"] == "llm_decided")
    root = {"span_id": "s0", "parent_id": None, "name": "invoke_agent",
            "start_ms": 0, "end_ms": clock,
            "attributes": {"gen_ai.operation.name": "invoke_agent",
                           "gen_ai.usage.input_tokens": in_tok,
                           "gen_ai.usage.output_tokens": out_tok,
                           "gen_ai.cost_usd": round((in_tok + out_tok) * PRICE_PER_TOKEN, 5)}}
    return [root] + children


print("build_spans ready (journal -> OTel-GenAI-shaped span tree).")

## Step 3 - Pretty-print the span tree

`print_span_tree` renders the spans the way a tracing UI would: the root on top with its aggregate tokens and cost, then each child indented under a tree branch with its time window and a per-span detail. An `llm` span shows which tool it decided and its token usage; an `execute_tool` span shows the tool name. This is the human-readable view; the JSONL below it is the machine view, and they are built from the exact same span dicts.

In [ ]:
def print_span_tree(spans):
    root = spans[0]
    a = root["attributes"]
    print(f"  invoke_agent  [{root['start_ms']}-{root['end_ms']}ms]  "
          f"in={a['gen_ai.usage.input_tokens']} out={a['gen_ai.usage.output_tokens']} "
          f"${a['gen_ai.cost_usd']:.5f}")
    kids = spans[1:]
    for i, s in enumerate(kids):
        branch = "  " + ("\u2514\u2500" if i == len(kids) - 1 else "\u251c\u2500")
        at = s["attributes"]
        if s["name"] == "llm":
            extra = f"decide {at['decided.tool']}  in={at['gen_ai.usage.input_tokens']} out={at['gen_ai.usage.output_tokens']}"
        else:
            extra = f"{at['gen_ai.tool.name']}"
        print(f"{branch} {s['name']:<13} [{s['start_ms']}-{s['end_ms']}ms]  {extra}")


print("print_span_tree ready.")

## Step 4 - Reconstruct metrics from the spans, not a side channel

The metrics are not gathered by a parallel counter sprinkled through the loop; they are READ BACK from the same spans we just emitted. `metrics_from_spans` derives the trajectory (the `execute_tool` spans, in order), the llm-call count (the `llm` spans), the token totals and cost (off the root), and the latency (the root's `end_ms`). Deriving metrics from the trace, rather than from a separate metrics path, means the dashboard and the trace can never disagree.

Note the trajectory is the tool sequence `search_policy -> process_refund`. `finish` is an llm decision, not a tool call, so it counts toward llm calls (3) but not the trajectory.

In [ ]:
def metrics_from_spans(spans):
    root = spans[0]
    tools = [s["attributes"]["gen_ai.tool.name"] for s in spans if s["name"] == "execute_tool"]
    llm_calls = sum(1 for s in spans if s["name"] == "llm")
    return {"trajectory": tools, "llm_calls": llm_calls,
            "input_tokens": root["attributes"]["gen_ai.usage.input_tokens"],
            "output_tokens": root["attributes"]["gen_ai.usage.output_tokens"],
            "cost_usd": root["attributes"]["gen_ai.cost_usd"],
            "latency_ms": root["end_ms"]}


print("metrics_from_spans ready (metrics are read back from the spans).")

## Step 5 - generate() and a real OTLP exporter: the real paths (reference only)

Same device as Parts 1-10. Offline, the frozen journal is the source of truth for everything in the output; the demo never calls `generate()`. On the real path the controller would be an LLM, and the usage it returns (input/output tokens, model id) is exactly what fills each `llm` span. That is the **controller = prompt + model** payoff from Part 1 made concrete: the `llm` span's `gen_ai.request.model` and token usage are where your model choice and thinking budget show up in the bill. Swap a bigger model or a larger thinking budget into `generate()` and you see it, line by line, in the span attributes.

SDK names and model ids move fast; only `generate()` (and a real OTLP exporter, the commented one-liner from Step 2) would need edits to go live.

In [ ]:
def generate(prompt):
    """REAL path: ask a hosted LLM; the usage it returns fills the llm span. Unused offline."""
    from openai import OpenAI
    client = OpenAI()                               # reads OPENAI_API_KEY
    resp = client.chat.completions.create(
        model="gpt-4o-mini",                        # a small chat model; check names
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content

# Anthropic / Claude alternative. Swap in for generate() above:
#
# def generate(prompt):
#     from anthropic import Anthropic
#     client = Anthropic()                            # reads ANTHROPIC_API_KEY
#     resp = client.messages.create(
#         model="claude-opus-4-8",                    # check current model names
#         max_tokens=1024,
#         messages=[{"role": "user", "content": prompt}],
#     )
#     return resp.content[0].text


if os.environ.get("OPENAI_API_KEY"):
    print("[trace] OPENAI_API_KEY set; real token usage would fill the spans, and a real OTLP "
          "exporter would ship them. Falling through to the frozen journal for reproducibility.")
else:
    print("[trace] no OPENAI_API_KEY; folding the frozen journal into spans (offline default)")

## Demo - one log, two views

Everything below runs fully offline. First the banner and the fold of the journal into spans, then the two views side by side: the Part 9 journal (durability) and the same run as a span tree plus JSONL (observability). They are the same bytes read two ways.

In [ ]:
bar = "=" * 72
print(bar)
print("SHIPPING IT  -  tracing, cost-per-success, and the core capstone")
print(bar)
if os.environ.get("OPENAI_API_KEY"):
    print("[trace] OPENAI_API_KEY set; real token usage would fill the spans, and a real OTLP "
          "exporter would ship them. Falling through to the frozen journal for reproducibility.")
else:
    print("[trace] no OPENAI_API_KEY; folding the frozen journal into spans (offline default)")

spans = build_spans(JOURNAL)

### One log, two views

The first view is the durability artifact from Part 9: the append-only journal, the thing that lets the agent crash and resume. The second view is the observability artifact: the SAME journal folded into a span tree and serialized as JSONL, the thing that lets you debug and account for the run. No second pipeline, no second store. One log, read two ways.

In [ ]:
# --- One log, two views: durability (journal) and observability (spans). ---
print("\n" + "-" * 72)
print("ONE LOG, TWO VIEWS. The Part 9 journal (durability)...")
print("-" * 72)
for e in JOURNAL:
    print("    " + json.dumps(e, sort_keys=True))

print("\n" + "-" * 72)
print("...folded into an OTel-GenAI-shaped span tree (observability). No otel SDK.")
print("-" * 72)
print_span_tree(spans)

print("\n  The same spans as JSONL (what a real OTLP exporter would ship):")
for s in spans:
    print("    " + json.dumps(s, sort_keys=True))

### Metrics from the spans

Now read the metrics back from those spans. The trajectory, the llm-call count, the token totals, the cost, and the latency all come from the span dicts, not from a side channel. The input-token note (`40 -> 70 -> 95`) is the transcript-economics sidebar made visible in the run.

In [ ]:
# --- Metrics from the spans. -------------------------------------------
print("\n" + bar)
print("METRICS reconstructed from the spans")
print(bar)
m = metrics_from_spans(spans)
print(f"  trajectory   : {' -> '.join(m['trajectory'])}")
print(f"  llm calls    : {m['llm_calls']}")
print(f"  tokens       : in={m['input_tokens']} out={m['output_tokens']} "
      f"(input grew 40 -> 70 -> 95: the re-sent transcript is the dominant cost)")
print(f"  cost         : ${m['cost_usd']:.5f}")
print(f"  latency      : {m['latency_ms']}ms (sum of per-span latencies)")

## Step 6 - Cost per success: the number that tells you if shipping it was worth it

A cheap agent that never succeeds is not cheap. In production you pay for the failures too. So the honest unit cost is not the cost of one happy run; it is **total cost across all runs divided by the number of successes**. It is always worse than a single happy run, because the failed runs still burned tokens.

Here three runs share the device: the refund for `ORD-3300` (success, `$0.00252`), a warranty lookup (success, `$0.00180`), and a hunt for a nonexistent discount (failure, `$0.00320` -- it spun until the Part 8 circuit breaker tripped). Total spend `$0.00752`, two successes, so cost-per-success is `$0.00376`: half again as much as the `$0.00252` a single happy run suggested. That gap is the price of the failure, and it is the number a budget owner actually cares about.

In [ ]:
# --- Cost per success across several runs. ------------------------------
print("\n" + bar)
print("COST PER SUCCESS  (you pay for the failures too)")
print(bar)
runs = [
    ("refund ORD-3300", True, m["cost_usd"]),
    ("warranty lookup", True, 0.00180),
    ("find a nonexistent discount", False, 0.00320),    # tripped the Part 8 breaker
]
total = sum(c for _, _, c in runs)
successes = sum(1 for _, ok, _ in runs if ok)
print(f"  {'task':<30}{'success':>9}{'cost':>12}")
print("  " + "-" * 49)
for name, ok, c in runs:
    print(f"  {name:<30}{('yes' if ok else 'no'):>9}{('$%.5f' % c):>12}")
print("  " + "-" * 49)
print(f"  total cost across {len(runs)} runs: ${total:.5f}; successes: {successes}")
print(f"  COST PER SUCCESS: ${total / successes:.5f}  "
      f"(worse than one happy run's ${m['cost_usd']:.5f}: the failed run still cost money)")

## Step 7 - The core capstone: every ring we added, Parts 1 to 11

This is where the core track ends, so name the whole thing. We started in Part 1 with the augmented LLM: a single model call ringed by typed tools, wrapped in the smallest loop. Every part since added exactly one ring. The checklist below is the full stack, bottom to top.

Storage for the journal (and therefore for both views) is about sixty lines: a JSONL file or a SQLite table. That is enough for this device. For real production durability the essay tours four systems with honest limits: **Temporal** (durable execution via event sourcing, heavyweight to operate), **DBOS** (durability in your Postgres, language-bound), the **Vercel Workflow DevKit** (durable steps on serverless, platform-shaped), and **LangGraph** (graph checkpointing, framework-coupled). They differ in where state lives and what they lock you into, but every one is the same idea as our journal: state is the fold over an append-only log.

The frontier track (Parts 12 to 17) opens this self-contained agent to the outside world: the protocol wire and multi-agent systems.

In [ ]:
# --- The capstone checklist. -------------------------------------------
print("\n" + bar)
print("THE CORE CAPSTONE  -  every ring we added to the augmented LLM, Parts 1 to 11")
print(bar)
rings = [
    "1  augmented-LLM loop with typed, validated tools",
    "2  robust execution: failure taxonomy, retries, idempotency (seed)",
    "3  planning: plan-and-execute / ReWOO / tool DAG (LLM-call + depth accounting)",
    "4  critic + error-triggered replanning over the DAG",
    "5  reflection + cross-trial Reflexion",
    "6  four typed memories the agent edits itself",
    "7  compaction + forgetting for the long haul",
    "8  budgets + loop detector + circuit breaker",
    "9  durable event journal + replay + effectively-once",
    "10 pause / approve / resume / steer (human-in-the-loop)",
    "11 observability: spans, cost-per-success  <- you are here",
]
for r in rings:
    print(f"    {r}")
print("\n  Storage is ~60 lines (a JSONL file or SQLite). For production durability the")
print("  essay tours Temporal, DBOS, the Vercel Workflow DevKit, and LangGraph, with")
print("  honest limits. The frontier track (Parts 12 to 17) opens this agent to the")
print("  protocol wire and multi-agent systems.")

print("\n" + bar)
print("Done. Observability is not a second system: the journal that makes the agent")
print("durable, read as spans, makes it debuggable and accountable. Cost-per-success is")
print("the number that tells you whether shipping it was worth it.")
print(bar)

## Wrap-up: the core track is complete

We close the core track exactly where Part 9 hinted: the journal that made the agent durable, read a second way, makes it observable. Nothing new had to be stood up.

- **A span tree, by hand.** One `invoke_agent` root over `llm` (chat) and `execute_tool` children, carrying `gen_ai.*`-STYLE attributes, emitted as JSONL. No otel SDK, because the conventions still move and the offline bar must hold; a real OTLP exporter is a commented one-liner.
- **Metrics from the spans.** Trajectory, llm calls, tokens, cost, and latency are read back from the trace, so the dashboard and the trace can never disagree.
- **Cost per success.** Total spend over successes, always worse than one happy run, because you pay for the failures too. It is the number that tells you whether shipping it was worth it.
- **One log, two views.** The durability artifact and the observability artifact are the same bytes. Storage is sixty lines; production durability is Temporal, DBOS, the Vercel Workflow DevKit, or LangGraph, all the same fold-over-a-log idea.

And the two sidebars that run through the bill: **transcript economics** (input tokens grow `40 -> 70 -> 95` because the transcript is re-sent every step, and prompt/KV caching is the lever back) and **controller = prompt + model** (the `llm` span's `request.model` and token usage are where model choice and thinking budget show up).

That completes the eleven rings of the core agent: a bare loop turned into a durable, pausable, observable, accountable system.

**Part 12 - the frontier track begins: MCP by hand.** So far the agent's tools have been local Python functions. Part 12 opens it to the protocol wire: we build the Model Context Protocol from first principles, so the agent can call tools it did not ship with, served by a process it does not control.